# NB20 — Quantum-Conditioned Multi-Agent Deliberation

**Quantum Brain Research Laboratory — Alejandro Reynoso**


## Purpose

Quantum routines return samples, paths or evidence portfolios—not executive judgment.
This notebook assigns the selected evidence to a small society of reasoning roles:
advocate, skeptic, causal analyst, historian, risk officer and weak-signal explorer.
A governed synthesizer then constructs a decision memorandum with explicit uncertainty.

To remain reproducible and API-free, the laboratory uses a deterministic text
synthesizer. A production version can replace `role_statement` with an approved LLM call
while preserving exactly the same evidence packets and audit records.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys

IN_COLAB = Path("/content").exists()
LAB_ROOT = Path("/content/Quantum_Brain_Lab") if IN_COLAB else Path.cwd() / "Quantum_Brain_Lab"
LAB_ROOT.mkdir(parents=True, exist_ok=True)
DEPS = LAB_ROOT / "_deps"

required = {
    "networkx": "networkx",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    DEPS.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--target", str(DEPS), *missing]
    )
    sys.path.insert(0, str(DEPS))

print(f"Laboratory root: {LAB_ROOT}")


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import random
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 271828
RNG = random.Random(SEED)
np.random.seed(SEED)

def stable_hash(value: Any) -> str:
    raw = json.dumps(value, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def cosine_words(a: str, b: str) -> float:
    wa = Counter(x.lower().strip(".,:;!?()") for x in a.split())
    wb = Counter(x.lower().strip(".,:;!?()") for x in b.split())
    keys = set(wa) | set(wb)
    if not keys:
        return 0.0
    va = np.array([wa[k] for k in keys], dtype=float)
    vb = np.array([wb[k] for k in keys], dtype=float)
    den = np.linalg.norm(va) * np.linalg.norm(vb)
    return float(va @ vb / den) if den else 0.0

def build_graph() -> nx.MultiDiGraph:
    """Synthetic governed investment-committee vault."""
    g = nx.MultiDiGraph(graph_version="QB-G0", title="Quantum Brain governed vault")
    nodes = {
        "AlphaBank": ("company", "Diversified bank with floating-rate loans, stable deposits, and legacy systems.", 0.00, 0.92),
        "BetaPayments": ("company", "Cloud-native payments platform with rapid growth, thin margins, and rich transaction data.", 0.20, 0.88),
        "ZetaCloud": ("company", "Enterprise cloud provider with recurring revenue and cyber concentration risk.", 0.05, 0.90),
        "GammaRetail": ("company", "Consumer retailer exposed to imported inventory and discretionary demand.", -0.10, 0.87),
        "DeltaLogistics": ("company", "Regional logistics network with pricing power and fuel exposure.", 0.10, 0.86),
        "EtaInsure": ("company", "Insurer benefiting from reinvestment yields while claims rise with inflation.", 0.00, 0.89),
        "RateShock": ("factor", "Policy rates rise sharply, increasing discount rates and funding costs.", -0.70, 0.96),
        "Regulation": ("factor", "Capital, data, conduct, and model-risk requirements tighten.", -0.40, 0.96),
        "CyberShock": ("factor", "A severe cyber event disrupts critical services and customer trust.", -0.80, 0.98),
        "ConsumerSlowdown": ("factor", "Real disposable income and discretionary demand weaken.", -0.70, 0.95),
        "FXShock": ("factor", "The peso depreciates and imported-input costs rise.", -0.60, 0.94),
        "AcquireBeta": ("decision", "AlphaBank considers acquiring BetaPayments.", 0.00, 0.95),
        "ExpandCredit": ("decision", "AlphaBank considers expanding unsecured consumer credit.", 0.00, 0.95),
        "MigrateCloud": ("decision", "The group considers migrating critical workloads to ZetaCloud.", 0.00, 0.95),
        "HedgeFX": ("decision", "GammaRetail considers increasing its foreign-exchange hedge ratio.", 0.00, 0.95),
        "E01": ("evidence", "Payments data can reduce fraud losses and improve cross-selling at AlphaBank.", 0.85, 0.91),
        "E02": ("evidence", "BetaPayments valuation is highly sensitive to higher discount rates.", -0.90, 0.94),
        "E03": ("evidence", "Legacy-control integration creates a material execution risk.", -0.80, 0.93),
        "E04": ("evidence", "Prior acquisitions performed better when product autonomy was preserved.", 0.70, 0.82),
        "E05": ("evidence", "Tighter data regulation increases the fixed cost of payments integration.", -0.75, 0.92),
        "E06": ("evidence", "A bank-payments data estate improves real-time risk detection.", 0.80, 0.89),
        "E07": ("evidence", "Unsecured credit losses rise nonlinearly in consumer slowdowns.", -0.95, 0.97),
        "E08": ("evidence", "Floating-rate assets initially benefit from higher rates.", 0.60, 0.88),
        "E09": ("evidence", "Deposit repricing can later compress the margin benefit.", -0.55, 0.90),
        "E10": ("evidence", "Independent model validation is required before credit expansion.", -0.65, 0.98),
        "E11": ("evidence", "Cloud migration reduces unit costs and improves analytic flexibility.", 0.75, 0.89),
        "E12": ("evidence", "Single-provider concentration can turn a cyber shock into a systemic outage.", -0.95, 0.97),
        "E13": ("evidence", "Workload segmentation contained a prior service disruption.", 0.65, 0.91),
        "E14": ("evidence", "Multi-cloud resilience reduces concentration but raises coordination cost.", 0.25, 0.85),
        "E15": ("evidence", "Layered FX hedges stabilize gross margin.", 0.75, 0.92),
        "E16": ("evidence", "Over-hedging destroys value if currency weakness reverses.", -0.65, 0.88),
        "E17": ("evidence", "FX collateral calls can create a temporary liquidity shock.", -0.55, 0.90),
        "WeakSignalA": ("signal", "A small merchant cohort is moving from cards to account-to-account payments.", 0.45, 0.67),
        "WeakSignalB": ("signal", "New cyber-insurance exclusions may transfer more outage risk to cloud clients.", -0.50, 0.69),
        "OutcomeAutonomy": ("outcome", "Preserved product autonomy accelerated customer migration in a prior deal.", 0.65, 0.94),
        "OutcomeCredit": ("outcome", "A prior downturn generated losses above the linear stress model.", -0.90, 0.96),
        "OutcomeCloud": ("outcome", "Segmentation reduced recovery time during an earlier outage.", 0.70, 0.95),
        "OutcomeHedge": ("outcome", "The hedge protected margin but triggered a collateral call.", 0.10, 0.95),
    }
    for node_id, (kind, text, polarity, reliability) in nodes.items():
        g.add_node(node_id, kind=kind, text=text, polarity=polarity,
                   reliability=reliability, status="authoritative",
                   source=f"source_{1 + len(node_id) % 9:02d}")
    edges = [
        ("AcquireBeta","AlphaBank","concerns"),("AcquireBeta","BetaPayments","concerns"),
        ("ExpandCredit","AlphaBank","concerns"),("MigrateCloud","ZetaCloud","concerns"),
        ("HedgeFX","GammaRetail","concerns"),("AlphaBank","RateShock","exposed_to"),
        ("AlphaBank","Regulation","exposed_to"),("BetaPayments","RateShock","exposed_to"),
        ("BetaPayments","Regulation","exposed_to"),("BetaPayments","ZetaCloud","depends_on"),
        ("ZetaCloud","CyberShock","exposed_to"),("GammaRetail","FXShock","exposed_to"),
        ("GammaRetail","ConsumerSlowdown","exposed_to"),("GammaRetail","DeltaLogistics","depends_on"),
        ("EtaInsure","RateShock","exposed_to"),("EtaInsure","ConsumerSlowdown","exposed_to"),
        ("RateShock","ConsumerSlowdown","causes"),("CyberShock","Regulation","causes"),
        ("FXShock","ConsumerSlowdown","causes"),
        ("E01","AcquireBeta","supports"),("E02","AcquireBeta","contradicts"),
        ("E03","AcquireBeta","contradicts"),("E04","AcquireBeta","supports"),
        ("E05","AcquireBeta","contradicts"),("E06","AcquireBeta","supports"),
        ("WeakSignalA","BetaPayments","informs"),("WeakSignalA","AcquireBeta","supports"),
        ("E04","OutcomeAutonomy","resulted_in"),("OutcomeAutonomy","AcquireBeta","supports"),
        ("E07","ExpandCredit","contradicts"),("E08","ExpandCredit","supports"),
        ("E09","ExpandCredit","contradicts"),("E10","ExpandCredit","contradicts"),
        ("E07","OutcomeCredit","resulted_in"),("OutcomeCredit","ExpandCredit","contradicts"),
        ("E11","MigrateCloud","supports"),("E12","MigrateCloud","contradicts"),
        ("E13","MigrateCloud","supports"),("E14","MigrateCloud","supports"),
        ("WeakSignalB","CyberShock","informs"),("WeakSignalB","MigrateCloud","contradicts"),
        ("E13","OutcomeCloud","resulted_in"),("OutcomeCloud","MigrateCloud","supports"),
        ("E15","HedgeFX","supports"),("E16","HedgeFX","contradicts"),
        ("E17","HedgeFX","contradicts"),("E15","OutcomeHedge","resulted_in"),
        ("OutcomeHedge","HedgeFX","supports"),
        ("E01","E06","corroborates"),("E03","Regulation","informs"),
        ("E05","Regulation","informs"),("E12","CyberShock","informs"),
        ("E17","FXShock","informs"),("E09","RateShock","informs"),
    ]
    for i, (u, v, rel) in enumerate(edges):
        g.add_edge(u, v, rel=rel, weight=0.65 + 0.35 * ((i % 7) / 6))
    return g

QUERIES = [
    {"id":"Q1","text":"Should AlphaBank acquire BetaPayments under higher rates and tighter regulation?",
     "target":"AcquireBeta","seeds":["RateShock","Regulation"],"risk":"high","expected":"caution"},
    {"id":"Q2","text":"Should AlphaBank expand unsecured credit during a consumer slowdown?",
     "target":"ExpandCredit","seeds":["ConsumerSlowdown","RateShock"],"risk":"high","expected":"caution"},
    {"id":"Q3","text":"How should critical workloads migrate to ZetaCloud under cyber risk?",
     "target":"MigrateCloud","seeds":["CyberShock","Regulation"],"risk":"high","expected":"qualified"},
    {"id":"Q4","text":"How should GammaRetail hedge foreign-exchange depreciation risk?",
     "target":"HedgeFX","seeds":["FXShock","ConsumerSlowdown"],"risk":"medium","expected":"qualified"},
]

def simple_graph(g: nx.MultiDiGraph) -> nx.Graph:
    h = nx.Graph()
    h.add_nodes_from(g.nodes(data=True))
    for u, v, d in g.edges(data=True):
        if h.has_edge(u, v):
            h[u][v]["weight"] += float(d.get("weight", 1.0))
        else:
            h.add_edge(u, v, weight=float(d.get("weight", 1.0)), rels={d.get("rel","related")})
    return h

def enumerate_paths(g: nx.MultiDiGraph, query: dict, cutoff: int = 5, cap: int = 80) -> list[list[str]]:
    h = simple_graph(g)
    starts = list(dict.fromkeys(query["seeds"] + [query["target"]]))
    destinations = [n for n, d in h.nodes(data=True)
                    if d.get("kind") in {"evidence","signal","outcome"}]
    paths = []
    for s in starts:
        for t in destinations:
            if s == t:
                continue
            try:
                for p in nx.all_simple_paths(h, s, t, cutoff=cutoff):
                    if query["target"] in p or any(seed in p for seed in query["seeds"]):
                        paths.append(p)
                        if len(paths) >= cap:
                            break
            except nx.NetworkXNoPath:
                pass
            if len(paths) >= cap:
                break
        if len(paths) >= cap:
            break
    unique = []
    seen = set()
    for p in paths:
        key = tuple(p)
        if key not in seen:
            seen.add(key); unique.append(p)
    return unique

def path_features(g: nx.MultiDiGraph, query: dict, path: list[str]) -> dict:
    attrs = [g.nodes[n] for n in path]
    text = " ".join(a.get("text","") for a in attrs)
    evidence = [a for a in attrs if a.get("kind") in {"evidence","outcome","signal"}]
    polarity = float(np.mean([a.get("polarity",0.0) for a in evidence])) if evidence else 0.0
    reliability = float(np.mean([a.get("reliability",0.5) for a in evidence])) if evidence else 0.5
    relevance = cosine_words(query["text"], text)
    kinds = {a.get("kind") for a in attrs}
    novelty = float(sum(a.get("kind") == "signal" for a in attrs) / max(1, len(path)))
    causal = float(any(g.nodes[n].get("kind") == "factor" for n in path)
                   and any(g.nodes[n].get("kind") in {"decision","outcome"} for n in path))
    contradiction = float(polarity < -0.15)
    support = float(polarity > 0.15)
    bridge = float(len(kinds) / 5.0)
    cost = len(path)
    score = (1.7*relevance + 0.9*reliability + 0.45*novelty + 0.40*causal
             + 0.35*bridge + 0.25*contradiction + 0.20*support - 0.06*cost)
    return {
        "relevance": relevance, "reliability": reliability, "novelty": novelty,
        "causal": causal, "contradiction": contradiction, "support": support,
        "bridge": bridge, "cost": cost, "polarity": polarity, "score": score,
    }

def build_catalog(g: nx.MultiDiGraph, queries: list[dict] = QUERIES) -> pd.DataFrame:
    rows = []
    for q in queries:
        for j, path in enumerate(enumerate_paths(g, q)):
            f = path_features(g, q, path)
            rows.append({"query_id":q["id"],"path_id":f"{q['id']}-P{j:03d}",
                         "path":" -> ".join(path),"nodes":path,**f})
    return pd.DataFrame(rows)

def json_graph(g: nx.MultiDiGraph) -> dict:
    return nx.node_link_data(g, edges="edges")

def load_graph(path: Path) -> nx.MultiDiGraph:
    return nx.node_link_graph(json.loads(path.read_text()), edges="edges",
                              directed=True, multigraph=True)

def ensure_baseline() -> tuple[nx.MultiDiGraph, pd.DataFrame]:
    graph_path = LAB_ROOT / "QB_baseline_graph.json"
    catalog_path = LAB_ROOT / "NB17_path_catalog.csv"
    if graph_path.exists() and catalog_path.exists():
        return load_graph(graph_path), pd.read_csv(catalog_path)
    g = build_graph()
    catalog = build_catalog(g)
    graph_path.write_text(json.dumps(json_graph(g), indent=2))
    catalog.assign(nodes=catalog["nodes"].apply(json.dumps)).to_csv(catalog_path, index=False)
    return g, catalog

def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 1e-15]
    return float(-(p*np.log(p)).sum()/np.log(max(2,len(p))))

def jensen_shannon(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p,dtype=float); q=np.asarray(q,dtype=float)
    p=p/p.sum(); q=q/q.sum(); m=0.5*(p+q)
    def kl(a,b):
        mask=a>1e-15
        return float(np.sum(a[mask]*np.log(a[mask]/np.maximum(b[mask],1e-15))))
    return math.sqrt(max(0.0,0.5*kl(p,m)+0.5*kl(q,m)))

def portfolio_metrics(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"paths":0,"mean_score":0,"contradiction_share":0,"support_share":0,
                "novelty":0,"node_coverage":0,"polarity_balance":0}
    nodes = set()
    for value in df["path"]:
        nodes.update(str(value).split(" -> "))
    return {
        "paths":len(df), "mean_score":float(df["score"].mean()),
        "contradiction_share":float(df["contradiction"].mean()),
        "support_share":float(df["support"].mean()),
        "novelty":float(df["novelty"].mean()), "node_coverage":len(nodes),
        "polarity_balance":float(1-abs(df["polarity"].mean())),
    }


In [ ]:
g,catalog=ensure_baseline()
portfolio_file=LAB_ROOT/"NB19_selected_portfolios.json"
if not portfolio_file.exists():
    # Independent fallback: use a diverse score-based portfolio.
    frame=catalog[catalog.query_id=="Q1"].nlargest(12,"score").copy()
    pos=frame[frame.polarity>0].head(2)
    neg=frame[frame.polarity<0].head(2)
    ids=pd.concat([pos,neg]).path_id.tolist()
    portfolios={"qaoa_feasible_mode":ids,"greedy":frame.head(4).path_id.tolist()}
else:
    portfolios=json.loads(portfolio_file.read_text())
q=QUERIES[0]


## 1. Evidence packets and deliberative roles


In [ ]:
ROLES={
    "advocate":lambda d:d.sort_values(["support","score"],ascending=False).head(2),
    "skeptic":lambda d:d.sort_values(["contradiction","score"],ascending=False).head(2),
    "causal_analyst":lambda d:d.sort_values(["causal","reliability"],ascending=False).head(2),
    "historian":lambda d:d[d["path"].str.contains("Outcome")].head(2),
    "risk_officer":lambda d:d.sort_values(["reliability","contradiction"],ascending=False).head(2),
    "weak_signal_explorer":lambda d:d.sort_values(["novelty","bridge"],ascending=False).head(2),
}

def role_statement(role:str, packet:pd.DataFrame) -> dict:
    if packet.empty:
        return {"role":role,"stance":"insufficient_evidence",
                "statement":"No qualifying path was present in the selected context.",
                "path_ids":[]}
    mean=float(packet.polarity.mean())
    stance="support" if mean>.15 else "challenge" if mean<-.15 else "qualified"
    evidence=" | ".join(packet.path.tolist())
    return {"role":role,"stance":stance,
            "statement":f"{role.replace('_',' ').title()} finds a {stance} signal. Evidence paths: {evidence}",
            "path_ids":packet.path_id.tolist(),
            "mean_reliability":float(packet.reliability.mean())}

def deliberate(method:str,ids:list[str]) -> dict:
    selected=catalog[catalog.path_id.isin(ids)].copy()
    contributions=[]
    for role,selector in ROLES.items():
        contributions.append(role_statement(role,selector(selected)))
    support=float(selected[selected.polarity>0].score.sum())
    challenge=float(selected[selected.polarity<0].score.sum())
    total=max(1e-9,support+challenge)
    net=(support-challenge)/total
    decision="proceed_with_conditions" if net>.18 else "do_not_proceed" if net<-.18 else "defer_and_resolve"
    confidence=min(.95,.45+.35*selected.reliability.mean()+.10*min(1,len(selected)/5))
    unresolved=[c["role"] for c in contributions if c["stance"]=="insufficient_evidence"]
    return {"query_id":q["id"],"method":method,"decision":decision,
            "confidence":float(confidence),"net_evidence":float(net),
            "unresolved_roles":unresolved,"contributions":contributions,
            "selected_path_ids":ids}


In [ ]:
deliberations=[deliberate(method,ids) for method,ids in portfolios.items()
               if method in {"exact","greedy","qaoa_feasible_mode"}]
rows=[{k:v for k,v in d.items() if k not in {"contributions","selected_path_ids"}}
      for d in deliberations]
deliberation_summary=pd.DataFrame(rows)
display(deliberation_summary.round(3))


## 2. Governed synthesis


In [ ]:
def render_memo(d:dict) -> str:
    lines=[
        f"# Decision Memorandum — {d['query_id']}",
        "",
        f"**Recommendation:** {d['decision'].replace('_',' ').title()}",
        f"**Calibrated confidence:** {d['confidence']:.2f}",
        f"**Evidence balance:** {d['net_evidence']:+.2f}",
        "",
        "## Deliberative record",
    ]
    for c in d["contributions"]:
        lines += [f"### {c['role'].replace('_',' ').title()}",c["statement"],""]
    lines += [
        "## Governance conditions",
        "- Treat every generated statement as analysis, not authoritative knowledge.",
        "- Preserve selected path identifiers and the graph version with the memorandum.",
        "- Escalate if high-reliability supporting and challenging evidence remain balanced.",
        "- Require human approval before any write-back to the governed vault.",
    ]
    return "\n".join(lines)

memo_dir=LAB_ROOT/"NB20_memos"; memo_dir.mkdir(exist_ok=True)
for d in deliberations:
    (memo_dir/f"{d['query_id']}_{d['method']}.md").write_text(render_memo(d))
with (LAB_ROOT/"NB20_deliberations.jsonl").open("w") as f:
    for d in deliberations:f.write(json.dumps(d)+"\n")
deliberation_summary.to_csv(LAB_ROOT/"NB20_deliberation_summary.csv",index=False)
print(render_memo(deliberations[-1])[:2400])


## 3. Shock sensitivity and disagreement


In [ ]:
shock_rows=[]
for shock_name,delta in {"base":0.0,"moderate_adverse":-0.12,"severe_adverse":-0.28}.items():
    for d in deliberations:
        shocked_net=d["net_evidence"]+delta
        decision="proceed_with_conditions" if shocked_net>.18 else "do_not_proceed" if shocked_net<-.18 else "defer_and_resolve"
        shock_rows.append({"shock":shock_name,"method":d["method"],
                           "net_evidence":shocked_net,"decision":decision,
                           "changed":decision!=d["decision"]})
shock_df=pd.DataFrame(shock_rows)
shock_df.to_csv(LAB_ROOT/"NB20_shock_sensitivity.csv",index=False)
display(shock_df)


In [ ]:
record={
    "notebook":"NB20","created_utc":utc_now(),"query_id":q["id"],
    "graph_hash":stable_hash(json_graph(g)),
    "portfolio_source":"NB19 or deterministic fallback",
    "roles":list(ROLES),"synthesis":"deterministic_auditable_prototype",
    "authority":"analysis_only_human_approval_required",
    "deliberations":len(deliberations)
}
(LAB_ROOT/"NB20_manifest.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


## Interpretation

The Quantum Brain is not complete when a quantum algorithm returns a promising sample.
Its cognitive value appears only after the sample has been translated into plural,
role-specific interpretations and then subjected to verification and authority controls.
NB21 closes the loop by testing governed recursive write-back and by separating
*quantum difference*, *quantum usefulness*, and *quantum advantage*.
